In [ ]:
from src.config import ProjectPath

In [ ]:
# [INFO] 1. SETUP CAU HINH & TIM DUONG DAN
data_name = "Tumors9"
n_features = 50

path = ProjectPath(data_name, n_features)
raw_path = path.raw_path

sfs_path = path.wrapper_dir / "Tumors9_SFS__1seed_3p_10max_raw.csv"
sklearn_sfs_path = path.wrapper_dir/ "Tumors9_sklearn_SFS__1seed_3p_automax_raw.csv"

anova_f_test_path = path.filter_dir / "Tumors9_anova_f_test_50features.csv"
chi_squared_path = path.filter_dir / "Tumors9_chi_squared_50features.csv"
correlation_path = path.filter_dir / "Tumors9_correlation_50features.csv"
mutual_information_path = path.filter_dir / "Tumors9_mutual_information_50features.csv"

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# [INFO] 2. DINH NGHIA HAM CHAM DIEM CHUYEN SAU (3 THUOC DO)
def evaluate_features(file_path, label):
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"[ERROR] Khong tim thay file: {file_path}")
        return None
        
    y = df.iloc[:, 0]
    X = df.iloc[:, 1:]
    
    # Luu y: Tumors9 co 9 class, de an toan tranh loi "n_splits" nhu tap Brain, dung cv=3
    cv_folds = 2 
    
    # Chia tap Train/Test (Do accuracy don thuan)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Khoi tao 2 Model 
    # (LogReg mac dinh la lbfgs ho tro multiclass rat tot, KHONG dung liblinear o day)
    log_model = LogisticRegression(max_iter=4000, random_state=42)
    tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
    
    # --- THUOC DO 1 & 2: Accuracy tren tap Test (Train 80, Test 20) ---
    log_model.fit(X_train, y_train)
    log_acc = log_model.score(X_test, y_test)
    
    tree_model.fit(X_train, y_train)
    tree_acc = tree_model.score(X_test, y_test)
    
    # --- THUOC DO 3: Cross Validation Score tren toan bo data ---
    log_cv = cross_val_score(log_model, X, y, cv=cv_folds, scoring='accuracy').mean()
    tree_cv = cross_val_score(tree_model, X, y, cv=cv_folds, scoring='accuracy').mean()
    
    return {
        "Method": label,
        "N_Feats": X.shape[1],
        "LogReg_Acc": round(log_acc, 4),
        "LogReg_CV": round(log_cv, 4),
        "Tree_Acc": round(tree_acc, 4),
        "Tree_CV": round(tree_cv, 4)
    }

# [INFO] 3. THUC THI VA XUAT KET QUA
print("[*] Dang cham diem cho SFS Custom...")
res_sfs = evaluate_features(sfs_path, "SFS_Custom")

print("[*] Dang cham diem cho Sklearn SFS...")
res_sklearn = evaluate_features(sklearn_sfs_path, "Sklearn_SFS")

print("[*] Dang cham diem cho ANOVA F-test...")
res_anova = evaluate_features(anova_f_test_path, "ANOVA_F")

print("[*] Dang cham diem cho Chi-squared...")
res_chi2 = evaluate_features(chi_squared_path, "Chi2")

print("[*] Dang cham diem cho Correlation...")
res_corr = evaluate_features(correlation_path, "Correlation")

print("[*] Dang cham diem cho Mutual Information...")
res_mi = evaluate_features(mutual_information_path, "Mutual_Info")

# Gom ket qua vao bang
results = [res for res in [res_sfs, res_sklearn, res_anova, res_chi2, res_corr, res_mi] if res is not None]

if results:
    df_results = pd.DataFrame(results)
    print("\n[RESULT] BANG SO SANH DO CHINH XAC (TUMORS9):")
    display(df_results) # Dung display de bang hien thi dep nhat trong Jupyter